In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.features.feature_engineer.apm_features import _detect_star_players
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.utils.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
No data available. Run getDict() first.
{}

Out Players:
No data available. Run getDict() first.
{}
No data available. Run getDict() first.
No data available. Run getDict() first.


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
p25 = pd.read_csv('data/raw/playoff_stats/P25.csv').sort_values(by='GAME_DATE')
s25 = pd.concat([s25, p25])
s25 = _detect_star_players(s25)

s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
s26 = pd.concat([s26, p26])
s26 = _detect_star_players(s26)

base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,TOP_PLAYER,SECOND_TOP_PLAYER,THIRD_TOP_PLAYER,IS_TOP_STAR,IS_TOP_1_STAR,ACTIVE_STARS_COUNT,TOP_STAR_ACTIVE,name
27518,NaN,NaN,868,2025-26,1629130,Duncan Robinson,Duncan,1610612765,DET,Detroit Pistons,42500105,2026-04-29,DET vs. ORL,W,24.953333,4,8,0.500,3,6,0.500,1,2,0.500,1,2,3,1,1,0,0,0,1,3,12,2,16.1,0,0,19.0,1,24:57,1,105.9,103.8,103.8,102.2,100.0,100.0,3.7,3.8,3.8,0.067,1.00,9.1,0.045,0.069,0.059,9.1,9.2,0.688,0.676,0.169,0.171,99.80,101.95,84.96,101.95,0.096,53,4.0,8.0,G,4.48,2.00,2.0,4.0,5.0,30.0,0.0,0.0,19.0,0.0,2.0,0.000,4.0,6.0,0.667,0.0,0.0,0.0,39,80,0.488,10,28,0.357,28,35,0.800,16,33,49,20,17.0,10,5,5,21,26,116,7.0,120.3,120.8,107.7,111.2,12.6,9.6,0.513,1.18,15.0,0.386,0.700,0.553,0.177,0.550,0.608,98.8,97.0,80.83,96,0.578,1610612753,ORL,Orlando Magic,38,80,0.475,17,38,0.447,16,30,0.533,8,25,33,21,16.0,12,5,5,26,21,109,-7.0,107.7,111.2,120.3,120.8,-12.6,-9.6,0.553,1.31,15.7,0.300,0.614,0.447,0.163,0.581,0.585,98.8,97.0,80.83,98,0.422,1,SF,31.0,NaN,NaN,-11.0,211.0,1,0.480898,0.040075,0.120224,1,3,Cade Cunningham,Jalen Duren,Paul Reed,0,0,2,1,Duncan Robinson
27519,NaN,NaN,869,2025-26,1628386,Jarrett Allen,Jarrett,1610612739,CLE,Cleveland Cavaliers,42500135,2026-04-29,CLE vs. TOR,W,25.106667,4,5,0.800,0,0,0.000,1,2,0.500,0,3,3,1,3,0,3,0,0,1,9,-4,20.1,0,0,19.0,1,25:06,1,113.1,114.3,114.3,122.8,125.9,125.9,-9.8,-11.6,-11.6,0.053,0.33,10.0,0.000,0.115,0.067,30.0,30.4,0.800,0.765,0.153,0.154,107.02,105.15,87.63,105.15,0.079,56,4.0,5.0,C,4.21,1.89,3.0,8.0,10.0,21.0,0.0,0.0,12.0,3.0,4.0,0.750,1.0,1.0,1.000,2.0,4.0,0.5,43,81,0.531,18,36,0.500,21,28,0.750,4,31,35,20,15.0,8,8,8,16,21,125,5.0,119.8,122.5,113.2,118.8,6.6,3.7,0.465,1.33,15.4,0.214,0.600,0.433,0.147,0.642,0.670,105.2,101.5,84.58,102,0.510,1610612761,TOR,Toronto Raptors,44,95,0.463,15,38,0.395,17,25,0.680,15,33,48,32,15.0,8,8,8,21,16,120,-5.0,113.2,118.8,119.8,122.5,-6.6,-3.7,0.727,2.13,20.8,0.400,0.786,0.567,0.149,0.542,0.566,105.2,101.5,84.58,101,0.490,1,C,27.0,NaN,NaN,-9.5,219.5,1,0.358471,0.039830,0.119490,1,0,Donovan Mitchell,James Harden,Jarrett Allen,1,0,3,1,Jarrett Allen
27520,NaN,NaN,845,2025-26,1629060,Rui Hachimu

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_odds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_odds = pd.json_normalize(data)

print("Loaded:", file.name)
team_odds.head()

Loaded: NBA_20260430_223641.json


,home_team,away_team,commence_time,bookmakers
0,Orlando Magic,Detroit Pistons,2026-05-01 23:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Toronto Raptors,Cleveland Cavaliers,2026-05-01 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,Houston Rockets,Los Angeles Lakers,2026-05-02 01:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
3,Boston Celtics,Philadelphia 76ers,2026-05-02 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
4,San Antonio Spurs,Minnesota Timberwolves,2026-05-05 01:30:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."


In [5]:
# Reverse map: full name → abbreviation (mirrors team_name_map from get_game_spread)
TEAM_NAME_TO_ABBREV = {
    'Atlanta Hawks': 'ATL', 'Boston Celtics': 'BOS', 'Brooklyn Nets': 'BKN',
    'Charlotte Hornets': 'CHA', 'Chicago Bulls': 'CHI', 'Cleveland Cavaliers': 'CLE',
    'Dallas Mavericks': 'DAL', 'Denver Nuggets': 'DEN', 'Detroit Pistons': 'DET',
    'Golden State Warriors': 'GSW', 'Houston Rockets': 'HOU', 'Indiana Pacers': 'IND',
    'LA Clippers': 'LAC', 'Los Angeles Lakers': 'LAL', 'Memphis Grizzlies': 'MEM',
    'Miami Heat': 'MIA', 'Milwaukee Bucks': 'MIL', 'Minnesota Timberwolves': 'MIN',
    'New Orleans Pelicans': 'NOP', 'New York Knicks': 'NYK', 'Oklahoma City Thunder': 'OKC',
    'Orlando Magic': 'ORL', 'Philadelphia 76ers': 'PHI', 'Phoenix Suns': 'PHX',
    'Portland Trail Blazers': 'POR', 'Sacramento Kings': 'SAC', 'San Antonio Spurs': 'SAS',
    'Toronto Raptors': 'TOR', 'Utah Jazz': 'UTA', 'Washington Wizards': 'WAS'
}

def get_game_context(base_df, player_name, team_odds, bookmaker_name='DraftKings'):
    pdf = base_df[base_df['PLAYER_NAME'] == player_name].sort_values('GAME_DATE')
    if pdf.empty:
        return f"Player '{player_name}' not found in base_df"

    team_name = pdf['TEAM_NAME'].iloc[-1]

    games = team_odds.to_dict('records') if isinstance(team_odds, pd.DataFrame) else team_odds

    game_data = next(
        (g for g in games if g.get('home_team') == team_name or g.get('away_team') == team_name),
        None
    )
    if game_data is None:
        return f"No game found for team '{team_name}'"

    bk = next(
        (b for b in game_data.get('bookmakers', []) if b.get('bookmaker') == bookmaker_name),
        None
    )
    if bk is None:
        return f"Bookmaker '{bookmaker_name}' not found for this game"

    # --- active_stars ---
    team_abbrev = TEAM_NAME_TO_ABBREV.get(team_name)
    stars = team3StarsPerTeam.get(team_abbrev, [])
    out = outPlayers.get(team_abbrev, [])
    active_stars = [s for s in stars if s not in out]

    result = {
        'player': player_name,
        'team': team_name,
        'opponent': game_data['away_team'] if game_data['home_team'] == team_name else game_data['home_team'],
        'is_home': game_data['home_team'] == team_name,
        'commence_time': game_data['commence_time'],
        'bookmaker': bookmaker_name,
        'active_stars': len(active_stars),   # count: 0, 1, 2, or 3
        'active_star_names': active_stars,   # optional: the names for debugging
        'spread': None,
        'spread_price': None,
        'total': None,
        'total_over_price': None,
        'total_under_price': None,
    }

    for market in bk.get('markets', []):
        if market['market_key'] == 'spreads':
            for outcome in market['outcomes']:
                if outcome['name'] == team_name:
                    result['spread'] = outcome['point']
                    result['spread_price'] = outcome['price']
        elif market['market_key'] == 'totals':
            for outcome in market['outcomes']:
                if outcome['name'] == 'Over':
                    result['total'] = outcome['point']
                    result['total_over_price'] = outcome['price']
                elif outcome['name'] == 'Under':
                    result['total_under_price'] = outcome['price']

    return result

def player_scenarios(df: pd.DataFrame, player_name: str, stat_name: str) -> dict:
    """
    Historical splits for a player covering the context signals
    that the base model does not capture:
        1. Active stars count  (roster context)
        2. Opponent pace       (game-speed context)
        3. Spread              (game-script / blowout risk)
        4. Home / Away         (venue context)

    Bayesian shrinkage toward the player's overall median:
        shrunk = (n * split_median + k * overall_median) / (n + k)
    k is a pseudo-count tuned per split type.
    """
    pdf = df[df['PLAYER_NAME'] == player_name].sort_values(by='GAME_DATE')
    overall_median = pdf[stat_name].median()
    total_n = len(pdf)

    def split_stats(subset, k=10):
        n = len(subset)
        if n == 0:
            return {'median': None, 'shrunk_median': None, 'delta': 0.0, 'hit_rate_vs_overall': None, 'n': 0}
        split_median = subset[stat_name].median()
        shrunk = (n * split_median + k * overall_median) / (n + k)
        hit_rate = (subset[stat_name] >= overall_median).mean()
        return {
            'median':              round(split_median, 4),
            'shrunk_median':       round(shrunk, 4),
            'delta':               round(shrunk - overall_median, 4),
            'hit_rate_vs_overall': round(hit_rate, 4),
            'n':                   n,
        }

    K_ACTIVE_STARS = 10
    K_PACE         = 10
    K_SPREAD       = 10
    K_HOME_AWAY    = 5

    # 1. Active stars count
    active_stars = {
        i: split_stats(pdf[pdf['ACTIVE_STARS_COUNT'] == i], k=K_ACTIVE_STARS)
        for i in [0, 1, 2, 3]
    }

    # 2. Game pace (proxied by Vegas total -- p25 / p75 of the merged dataset)
    opp_pace = {
        'high_pace':   split_stats(pdf[pdf['GAME_TOTAL'] > 234.5], k=K_PACE),
        'middle_pace': split_stats(pdf[(pdf['GAME_TOTAL'] >= 225.0) & (pdf['GAME_TOTAL'] <= 235.0)], k=K_PACE),
        'low_pace':    split_stats(pdf[pdf['GAME_TOTAL'] < 225.0], k=K_PACE),
    }

    # 3. Spread
    spread = {
        'favorite':       split_stats(pdf[(pdf['TEAM_SPREAD'] < 0)], k=K_SPREAD),
        'underdog':       split_stats(pdf[(pdf['TEAM_SPREAD'] > 0)], k=K_SPREAD),
    }

    # 4. Home / Away
    home_away = {
        'home': split_stats(pdf[pdf['IS_HOME'] == 1], k=K_HOME_AWAY),
        'away': split_stats(pdf[pdf['IS_HOME'] == 0], k=K_HOME_AWAY),
    }

    overall_iqr = (
        pdf[stat_name].quantile(0.75) - pdf[stat_name].quantile(0.25)
        if total_n > 0 else None
    )

    return {
        'player':         player_name,
        'stat':           stat_name,
        'overall_median': round(overall_median, 4) if pd.notna(overall_median) else None,
        'overall_iqr':    round(overall_iqr, 4) if overall_iqr is not None and pd.notna(overall_iqr) else None,
        'total_games':    total_n,
        'active_stars':   active_stars,
        'opp_pace':       opp_pace,
        'spread':         spread,
        'home_away':      home_away,
    }

# Usage
scenarios = player_scenarios(base_df, 'Mikal Bridges', 'MIN')
result = get_game_context(base_df, 'Payton Pritchard', team_odds)
result

{'player': 'Payton Pritchard',
 'team': 'Boston Celtics',
 'opponent': 'Philadelphia 76ers',
 'is_home': True,
 'commence_time': Timestamp('2026-05-02 23:40:00+0000', tz='UTC'),
 'bookmaker': 'DraftKings',
 'active_stars': 3,
 'active_star_names': ['Jaylen Brown', 'Jayson Tatum', 'Derrick White'],
 'spread': -8.5,
 'spread_price': -105,
 'total': 206.5,
 'total_over_price': -110,
 'total_under_price': -110}

In [6]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = s26
ast_df = s26
reb_df = s26
min_df = s26

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
# lines_dfs = lines_dfs[lines_dfs['COMMENCE_TIME'] == current_date]
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
# lines_us = lines_us[lines_us['COMMENCE_TIME'] == current_date]
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-04-30 22:36:13
US latest pull: 2026-04-30 22:36:41


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,Desmond Bane,Over,19.5,-137,2026-05-01,2026-05-01T05:35:33Z,2026-04-30 22:36:13
1,Underdog,player_points,Desmond Bane,Under,19.5,-137,2026-05-01,2026-05-01T05:35:33Z,2026-04-30 22:36:13
2,Underdog,player_points,Tobias Harris,Over,17.5,-137,2026-05-01,2026-05-01T05:35:33Z,2026-04-30 22:36:13
3,Underdog,player_points,Tobias Harris,Under,17.5,-137,2026-05-01,2026-05-01T05:35:33Z,2026-04-30 22:36:13
4,Underdog,player_points,Jalen Duren,Over,13.5,-137,2026-05-01,2026-05-01T05:35:33Z,2026-04-30 22:36:13


### Load my models

In [7]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb_2026-01-02.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb_2026-01-01.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb_2026-01-01.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb_2026-01-01.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [8]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
pts_preds.head(10)

[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Desmond Bane,PTS,27.80,35.93,40.83,0.3074,0.5291,0.7898,8.54,19.01,32.25,"[0.6616166458912647, 0.6411062225015713, 0.562..."
1,Tobias Harris,PTS,23.22,31.85,39.16,0.2213,0.4762,0.7573,5.14,15.17,29.65,"[0.4389402727700267, 0.620884289746002, 0.4630..."
2,Jalen Duren,PTS,22.93,31.69,38.48,0.3013,0.5018,0.8239,6.91,15.90,31.70,"[0.6, 0.6183115338882283, 0.9544008483563096, ..."
3,Anthony Black,PTS,19.08,26.41,33.75,0.1867,0.4277,0.7341,3.56,11.29,24.78,"[0.2227998514667657, 0.3024054982817869, 0.262..."
4,Scottie Barnes,PTS,29.81,37.49,41.95,0.3259,0.5451,0.8267,9.72,20.43,34.68,"[0.6304248515303793, 0.8108108108108107, 0.180..."
5,Donovan Mitchell,PTS,28.09,36.16,41.49,0.3819,0.6154,0.9591,10.73,22.25,39.80,"[0.5607476635514019, 0.6923076923076923, 1.209..."
6,James Harden,PTS,29.53,38.70,42.53,0.3840,0.6030,0.9291,11.34,23.34,39.52,"[0.5514705882352942, 0.6995515695067266, 0.423..."
7,Brandon Ingram,PTS,23.26,32.93,39.55,0.3113,0.5268,0.8008,7.24,17.35,31.67,"[0.2200488997555012, 0.503731343283582, 0.4255..."
8,Evan Mobley,PTS,24.88,33.42,38.74,0.2961,0.5098,0.7515,7.37,17.04,29.12,"[0.5914567360350492, 0.5845254576219043, 0.261..."
9,Dennis Schroder,PTS,15.79,19.73,25.20,0.2029,0.4618,0.7458,3.20,9.11,18.80,"[0.3769633507853403, 0.1442654484251022, 0.401..."


### Get Line Probabilities

In [9]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
87,VJ Edgecombe,PTS,11.5,31.80,39.84,43.41,6.21,17.43,32.09,0.616,0.384
62,Scottie Barnes,PTS,20.5,29.81,37.49,41.95,9.72,20.43,34.68,0.533,0.467
84,Joel Embiid,PTS,26.5,23.19,29.37,36.59,9.61,19.71,36.61,0.423,0.577
81,Jaxson Hayes,PTS,5.5,15.68,18.39,25.02,2.08,7.68,18.19,0.634,0.366
39,Evan Mobley,REB,9.0,24.88,33.42,38.74,3.76,7.94,14.15,0.451,0.549
36,Paolo Banchero,REB,8.5,31.58,39.85,43.97,3.97,8.67,14.68,0.621,0.379
67,Dennis Schroder,PTS,6.5,15.79,19.73,25.20,3.20,9.11,18.80,0.799,0.201
82,Jayson Tatum,PTS,24.5,25.66,29.76,39.29,9.76,17.79,36.34,0.324,0.676
59,Tobias Harris,PTS,17.5,23.22,31.85,39.16,5.14,15.17,29.65,0.535,0.465
27,Marcus Smart,REB,2.5,23.50,32.76,39.88,0.86,3.36,8.45,0.619,0.381


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
23,Dean Wade,REB,3.5,16.38,21.27,28.83,1.16,3.65,8.63,0.636,0.364,REB,Underdog,Toronto Raptors,-4.0,219.5,112.1,5.0,99.22,21.0,-112.0,-110.0,0.528,0.524,4.1,4.5,1.66,0.6,1.0,-0.361,0.641,0.359,21.33,-31.46,0.6,0.6,0.60,0.56,22.35,3.63,0.09,0.05,4.56,9.0
71,Sam Merrill,PTS,7.5,17.65,22.29,28.63,3.17,10.15,21.11,0.640,0.360,PTS,Underdog,Toronto Raptors,-4.0,219.5,112.1,5.0,99.22,21.0,-111.0,-115.0,0.526,0.535,8.7,8.0,5.48,1.2,0.5,-0.219,0.587,0.413,11.58,-22.79,0.4,0.5,0.67,0.54,23.91,4.19,0.14,0.06,7.80,10.0
84,Joel Embiid,PTS,26.5,23.19,29.37,36.59,9.61,19.71,36.61,0.423,0.577,PTS,Underdog,Boston Celtics,7.8,206.5,111.7,4.0,95.58,30.0,-137.0,-137.0,0.578,0.578,28.8,28.0,4.98,4.3,3.5,-0.863,0.806,0.194,39.43,-66.44,0.8,0.9,0.87,0.64,33.50,4.16,0.34,0.04,20.83,6.0
85,Paul George,PTS,15.5,27.28,35.20,42.07,7.45,17.23,31.62,0.679,0.321,PTS,Underdog,Boston Celtics,7.8,206.5,111.7,4.0,95.58,30.0,-114.0,-108.0,0.533,0.519,16.1,16.5,4.23,0.6,1.0,-0.142,0.556,0.444,4.37,-14.49,1.0,0.8,0.87,0.51,32.38,7.16,0.21,0.04,16.43,7.0
33,Payton Pritchard,REB,3.5,22.62,29.65,35.26,1.06,3.56,8.54,0.464,0.536,REB,Underdog,Philadelphia 76ers,-7.8,206.5,114.4,17.0,100.40,15.0,122.0,-162.0,0.450,0.618,3.2,3.0,1.14,-0.3,-0.5,0.263,0.396,0.604,-12.09,-2.32,0.6,0.4,0.40,0.52,30.65,3.59,0.20,0.05,3.17,12.0


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
88,Tyrese Maxey,PTS,23.5,31.63,40.85,45.52,11.44,24.21,42.68,0.653,0.346,PTS,PrizePicks,Boston Celtics,7.8,206.5,111.7,4.0,95.58,30.0,-118.0,-103.0,0.541,0.507,24.3,23.5,5.19,0.8,0.0,-0.154,0.561,0.439,3.64,-13.48,0.6,0.5,0.53,0.69,37.30,4.89,0.28,0.05,27.58,12.0
70,Ja'Kobe Walter,PTS,9.5,21.39,30.96,38.27,2.42,12.99,27.37,0.605,0.395,PTS,PrizePicks,Cleveland Cavaliers,4.0,219.5,114.1,15.0,100.70,13.0,100.0,-110.0,0.500,0.524,9.9,10.0,6.64,0.4,0.5,-0.060,0.524,0.476,4.80,-9.13,0.4,0.5,0.47,0.36,28.07,3.28,0.13,0.03,6.30,10.0
23,Dean Wade,REB,3.5,16.38,21.27,28.83,1.16,3.65,8.63,0.636,0.364,REB,PrizePicks,Toronto Raptors,-4.0,219.5,112.1,5.0,99.22,21.0,-112.0,-110.0,0.528,0.524,4.1,4.5,1.66,0.6,1.0,-0.361,0.641,0.359,21.33,-31.46,0.6,0.6,0.60,0.56,22.35,3.63,0.09,0.05,4.56,9.0
79,Tari Eason,PTS,13.5,21.75,31.68,39.19,4.07,13.77,28.37,0.605,0.395,PTS,PrizePicks,Los Angeles Lakers,-4.0,206.5,115.5,20.0,99.22,22.0,100.0,-114.0,0.500,0.533,11.8,13.5,7.35,-1.7,0.0,0.231,0.409,0.591,-18.20,10.94,0.6,0.5,0.60,0.33,28.32,7.19,0.17,0.07,12.22,9.0
75,Alperen Sengun,PTS,20.5,29.88,38.23,43.92,9.70,20.73,35.63,0.550,0.450,PTS,PrizePicks,Los Angeles Lakers,-4.0,206.5,115.5,20.0,99.22,22.0,-111.0,-107.0,0.526,0.517,19.0,19.0,6.88,-1.5,-1.5,0.218,0.414,0.586,-21.30,13.37,0.2,0.3,0.40,0.41,34.86,6.37,0.25,0.04,19.33,9.0


In [12]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
5,Derrick White,AST,3.5,24.23,28.79,37.21,1.17,3.30,7.16,0.425,0.575,AST,Betr DFS,Philadelphia 76ers,-7.8,206.5,114.4,17.0,100.40,15.0,-114.0,-103.0,0.533,0.507,3.7,3.0,1.70,0.2,-0.5,-0.118,0.547,0.453,2.68,-10.72,0.2,0.4,0.53,0.71,33.05,6.46,0.15,0.06,4.69,13.0
82,Jayson Tatum,PTS,24.5,25.66,29.76,39.29,9.76,17.79,36.34,0.324,0.676,PTS,Betr DFS,Philadelphia 76ers,-7.8,206.5,114.4,17.0,100.40,15.0,-108.0,-118.0,0.519,0.541,24.1,24.0,2.73,-0.4,-0.5,0.147,0.442,0.558,-14.87,3.09,0.6,0.4,0.40,0.52,37.03,3.75,0.27,0.04,26.67,9.0
35,Jalen Duren,REB,9.5,22.93,31.69,38.48,4.35,9.77,17.70,0.565,0.435,REB,Betr DFS,Orlando Magic,-3.5,210.0,113.6,13.0,100.56,14.0,-137.0,-137.0,0.578,0.578,9.0,9.0,1.94,0.0,0.0,0.000,0.500,0.500,-13.50,-13.50,0.0,0.1,0.33,0.57,29.71,3.77,0.20,0.05,9.08,12.0
100,Max Strus,PTS,9.5,18.41,23.35,29.07,2.72,9.99,21.39,0.479,0.521,PTS,Betr DFS,Toronto Raptors,-4.0,219.5,112.1,5.0,99.22,21.0,105.0,-118.0,0.488,0.541,10.2,8.0,8.31,0.7,-1.5,-0.084,0.533,0.467,9.27,-13.72,0.4,0.4,0.47,0.47,24.57,3.90,0.17,0.05,11.43,7.0
63,Donovan Mitchell,PTS,26.5,28.09,36.16,41.49,10.73,22.25,39.80,0.420,0.580,PTS,Betr DFS,Toronto Raptors,-4.0,219.5,112.1,5.0,99.22,21.0,-115.0,-103.0,0.535,0.507,25.4,27.5,9.09,-1.1,1.0,0.121,0.452,0.548,-15.50,8.00,0.4,0.5,0.53,0.49,33.11,3.78,0.30,0.06,23.20,10.0


In [13]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
42,Amen Thompson,REB,7.5,32.05,41.13,45.26,2.95,7.10,12.20,0.398,0.602,REB,DraftKings Pick6,Los Angeles Lakers,-4.0,206.5,115.5,20.0,99.22,22.0,102.0,-110.0,0.495,0.524,7.0,7.0,2.79,-0.5,-0.5,0.179,0.429,0.571,-13.34,9.01,0.2,0.4,0.53,0.52,41.04,4.68,0.21,0.03,8.50,10.0
37,Ausar Thompson,REB,7.5,23.00,31.35,38.48,2.54,6.43,11.91,0.565,0.435,REB,DraftKings Pick6,Orlando Magic,-3.5,210.0,113.6,13.0,100.56,14.0,-104.0,-125.0,0.510,0.556,6.9,6.0,3.45,-0.6,-1.5,0.174,0.431,0.569,-15.46,2.42,0.8,0.4,0.33,0.22,28.56,5.35,0.16,0.05,8.09,11.0
38,Jalen Suggs,REB,4.5,26.25,35.05,40.64,1.16,4.19,9.19,0.405,0.595,REB,DraftKings Pick6,Detroit Pistons,3.5,210.0,108.9,2.0,99.88,19.0,137.0,-149.0,0.422,0.598,3.4,3.5,2.17,-1.1,-1.0,0.507,0.306,0.694,-27.48,15.98,0.2,0.3,0.33,0.34,32.48,5.63,0.20,0.06,3.17,12.0
56,Duncan Robinson,REB,2.5,20.45,27.23,34.13,0.74,3.01,7.44,0.526,0.474,REB,DraftKings Pick6,Orlando Magic,-3.5,210.0,113.6,13.0,100.56,14.0,120.0,-139.0,0.455,0.582,2.2,2.0,1.87,-0.3,-0.5,0.160,0.436,0.564,-4.08,-3.02,0.4,0.4,0.33,0.44,26.36,3.02,0.17,0.05,2.42,12.0
70,Ja'Kobe Walter,PTS,9.5,21.39,30.96,38.27,2.42,12.99,27.37,0.605,0.395,PTS,DraftKings Pick6,Cleveland Cavaliers,4.0,219.5,114.1,15.0,100.70,13.0,100.0,-110.0,0.500,0.524,9.9,10.0,6.64,0.4,0.5,-0.060,0.524,0.476,4.80,-9.13,0.4,0.5,0.47,0.36,28.07,3.28,0.13,0.03,6.30,10.0


In [14]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
21,James Harden,REB,4.5,29.53,38.70,42.53,1.65,4.94,9.97,0.544,0.456,REB,Underdog,Toronto Raptors,-4.0,219.5,112.1,5.0,99.22,21.0,-105.0,-110.0,0.512,0.524,3.3,3.0,2.71,-1.2,-1.5,0.443,0.329,0.671,-35.77,28.10,0.4,0.3,0.47,0.58,33.57,4.37,0.28,0.05,5.12,8.0
57,Ja'Kobe Walter,REB,3.5,21.39,30.96,38.27,0.87,3.35,8.79,0.418,0.582,REB,DraftKings Pick6,Cleveland Cavaliers,4.0,219.5,114.1,15.0,100.70,13.0,134.0,-146.0,0.427,0.593,2.9,2.0,2.33,-0.6,-1.5,0.258,0.398,0.602,-6.87,1.43,0.2,0.2,0.40,0.30,28.07,3.28,0.13,0.03,2.60,10.0
65,Brandon Ingram,PTS,18.5,23.26,32.93,39.55,7.24,17.35,31.67,0.408,0.592,PTS,Underdog,Cleveland Cavaliers,4.0,219.5,114.1,15.0,100.70,13.0,-112.0,-113.0,0.528,0.531,17.7,16.5,10.32,-0.8,-2.0,0.078,0.469,0.531,-11.23,0.09,0.2,0.4,0.33,0.64,31.49,7.59,0.23,0.06,16.44,9.0
72,Collin Murray-Boyles,PTS,11.5,17.56,23.30,30.38,2.98,10.69,22.47,0.594,0.406,PTS,PrizePicks,Cleveland Cavaliers,4.0,219.5,114.1,15.0,100.70,13.0,-114.0,-112.0,0.533,0.528,13.2,14.5,6.23,1.7,3.0,-0.273,0.608,0.392,14.13,-25.80,0.8,0.7,0.60,0.32,23.16,4.41,0.18,0.07,12.29,7.0
21,James Harden,REB,4.5,29.53,38.70,42.53,1.65,4.94,9.97,0.544,0.456,REB,Betr DFS,Toronto Raptors,-4.0,219.5,112.1,5.0,99.22,21.0,-105.0,-110.0,0.512,0.524,3.3,3.0,2.71,-1.2,-1.5,0.443,0.329,0.671,-35.77,28.10,0.4,0.3,0.47,0.58,33.57,4.37,0.28,0.05,5.12,8.0


### Get top EVs for 2 legs

In [15]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 61  |  Pairs: 24  |  Slate: 3  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks.json


In [16]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 46  |  Pairs: 14  |  Slate: 2  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 52  |  Pairs: 13  |  Slate: 2  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings.json


In [18]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 56  |  Pairs: 25  |  Slate: 2  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr.json


### Top EVs for 3 Legs

In [19]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 61  |  Triples: 171  |  Slate: 2  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks_3leg.json


In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 46  |  Triples: 52  |  Slate: 1  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 56  |  Triples: 152  |  Slate: 2  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr_3leg.json


In [22]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 52  |  Triples: 64  |  Slate: 2  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings_3leg.json
